<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/07_Taller_CNN_Texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: redes convolucionales para texto</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `15_CNN_Texto.ipynb` con **otra tarea y otra unidad de entrada**: en lugar de clasificar sentimiento con palabras, haremos **identificación de idioma** con **caracteres**. Usaremos el corpus [`papluca/language-identification`](https://huggingface.co/datasets/papluca/language-identification): 90 000 oraciones en 20 idiomas (reseñas de productos y oraciones de corpus de inferencia y similitud semántica), con 70 000 para entrenamiento, 10 000 para validación y 10 000 para prueba.

La identificación de idioma es el primer paso de muchos sistemas de PLN (motores de búsqueda, moderación, traducción) y un caso ideal para las CNN de caracteres: los idiomas se distinguen por N-gramas de letras característicos (*-ción*, *-ção*, *-zione*, *th*, *ß*), que es justamente lo que detecta un filtro convolucional.

Enfocamos:

- la convolución 1D con *stride*, dilatación y relleno, y el campo receptivo,
- una CNN de caracteres frente a una línea base fuerte de N-gramas de caracteres,
- filtros de varios anchos y su interpretación,
- la comparación con una LSTM bidireccional en exactitud, velocidad y robustez ante textos cortos.

Los modelos son pequeños para que el taller se pueda ejecutar en CPU (o en Colab). Resuelve cada ejercicio en la celda indicada y responde las preguntas en celdas markdown. Usa el conjunto de prueba **una sola vez**, al final del ejercicio 4.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Convoluciones 1D</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Clasificación con CNN</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Filtros de varios tamaños</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Comparación CNN vs LSTM</span>](#ejercicio-4)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow datasets

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # oculta mensajes informativos de TensorFlow

import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # fija las semillas de Python, NumPy y TensorFlow

import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

idiomas = load_dataset("papluca/language-identification")
df_train, df_val, df_test = (idiomas[s].to_pandas() for s in ["train", "validation", "test"])
CLASES = sorted(df_train.labels.unique())
c2i = {c: i for i, c in enumerate(CLASES)}
X_train_txt, X_val_txt, X_test_txt = df_train.text.tolist(), df_val.text.tolist(), df_test.text.tolist()
y_train, y_val, y_test = (df.labels.map(c2i).to_numpy() for df in (df_train, df_val, df_test))
print(f"Entrenamiento: {len(X_train_txt)} | Validación: {len(X_val_txt)} | Prueba: {len(X_test_txt)}")
print("Idiomas:", CLASES)

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Convoluciones 1D</h2>

**Objetivo:** generalizar la convolución 1D del cuaderno 15 con *stride*, dilatación y relleno, y medir empíricamente el campo receptivo de una red apilada.

1. Implementa en NumPy `conv1d(X, W, b, stride=1, dilatacion=1, relleno="valid")`, con `X` de forma `(T, d)`, `W` de forma `(k, d, m)` y `b` de forma `(m,)`, **sin** función de activación. Con `relleno="same"` y `stride=1`, la salida debe tener longitud $T$: rellena con ceros $\lfloor p/2 \rfloor$ posiciones a la izquierda y el resto a la derecha, donde $p = d_{\text{il}}(k - 1)$ es el relleno total.
2. Comprueba que coincide con `layers.Conv1D` (con `activation=None`) en al menos estas cuatro configuraciones: `k=3` básica; `k=3, strides=2`; `k=3, dilation_rate=2`; `k=4, padding="same"`. Verifica también la fórmula de la longitud de salida con relleno `valid`: $L = \lfloor (T - d_{\text{il}}(k-1) - 1)/s \rfloor + 1$.
3. El campo receptivo de $L$ capas con anchos $k_\ell$, dilataciones $d_\ell$ y *strides* $s_\ell$ es $r = 1 + \sum_\ell (k_\ell - 1)\, d_\ell \prod_{i<\ell} s_i$. Calcúlalo para una red de tres `Conv1D` con `k=3` y `strides=2` en cada capa.
4. Mide el campo receptivo **empíricamente** para esa red (con pesos aleatorios): calcula la salida para una entrada aleatoria, modifica solo la posición 40 de la entrada y cuenta qué posiciones de la salida cambian. Luego haz lo contrario: para la salida en la posición 5, encuentra qué posiciones de la entrada la afectan. ¿Coincide con la fórmula?

In [ ]:
# Escribe tu código aquí
def conv1d(X, W, b, stride=1, dilatacion=1, relleno="valid"):
    pass

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Clasificación con CNN</h2>

**Objetivo:** entrenar una CNN de caracteres para identificar el idioma y compararla con una línea base de N-gramas de caracteres.

1. Explora los datos: ¿cuántos ejemplos hay por idioma? ¿Cuál es la distribución de la longitud en **caracteres**? Elige `LONGITUD` como el percentil 95 de la longitud en entrenamiento.
2. **Línea base:** entrena un `TfidfVectorizer` de N-gramas de **caracteres** (`analyzer="char_wb"`, `ngram_range=(1, 3)`, como máximo 100 000 rasgos) con una regresión logística. Reporta la exactitud en validación.
3. Crea una capa `TextVectorization` a nivel de caracteres (`split="character"`, `standardize="lower"`) con un vocabulario de como máximo 2 000 caracteres y `output_sequence_length=LONGITUD`. ¿Cuántos caracteres distintos hay en entrenamiento? ¿Por qué hay tantos?
4. Entrena una CNN de caracteres: `Embedding` de dimensión 32, `Conv1D` de 128 filtros de ancho 5 con ReLU, `GlobalMaxPooling1D`, `Dropout(0.3)` y una capa de salida softmax de 20 clases. Usa parada temprana con paciencia 2 y como máximo 15 épocas. Reporta la exactitud en validación y el tiempo de entrenamiento.
5. ¿Qué pares de idiomas se confunden más? Muestra los 5 pares más frecuentes en la matriz de confusión de validación y algunos ejemplos mal clasificados.

In [ ]:
# Escribe tu código aquí
LONGITUD = None

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Filtros de varios tamaños</h2>

**Objetivo:** construir una CNN multifiltro (Kim, 2014) de caracteres e interpretar qué N-gramas detectan sus filtros para cada idioma.

1. Con la API funcional, construye una CNN con filtros de anchos 2, 3, 4 y 5 en paralelo (64 filtros de cada ancho, cada rama con su `GlobalMaxPooling1D`), concatena los resultados y añade `Dropout(0.5)` y la capa de salida. Ponles nombre a las capas convolucionales y a la de salida para poder recuperarlas. Compárala con la CNN del ejercicio 2.
2. Para el **español** (`es`) y el **portugués** (`pt`), encuentra los 3 filtros que más **contribuyen** a esa clase: su peso hacia la clase en la capa de salida multiplicado por su activación media (después del *pooling*) en los textos de ese idioma. Muestra los 6 N-gramas que más los activan **dentro de los textos de ese idioma** (sin repetir). Haz lo mismo para un idioma con otro alfabeto (por ejemplo `ru`). ¿Qué pasa si buscas los N-gramas en los textos de todos los idiomas?
3. ¿Los N-gramas tienen sentido lingüístico? ¿Qué rasgos distinguen el español del portugués?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Comparación CNN vs LSTM</h2>

**Objetivo:** comparar una CNN y una LSTM bidireccional de caracteres en exactitud, costo y robustez ante textos cortos, y evaluar el modelo elegido en prueba.

1. Entrena una LSTM bidireccional de caracteres (`Embedding(V, 32, mask_zero=True)`, `Bidirectional(LSTM(64))`, `Dropout(0.3)` y la capa de salida), con el mismo procedimiento. Añádela a la tabla comparativa.
2. **Robustez ante textos cortos:** en un sistema real, a menudo hay que identificar el idioma de un mensaje de pocas palabras. Recorta los textos de validación a sus primeros 10, 20, 40 y 80 caracteres y grafica la exactitud de los cuatro modelos (TF-IDF, CNN, CNN multifiltro, BiLSTM) en función de la longitud.
3. Elige el mejor modelo según **validación** (considera también el costo) y evalúalo **una sola vez** en prueba. Reporta la exactitud por idioma.
4. ¿Qué modelo pondrías en producción para identificar el idioma de millones de mensajes por día? Justifica.

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Kim, Y. (2014). Convolutional Neural Networks for Sentence Classification. *EMNLP*.
- Zhang, X., Zhao, J. & LeCun, Y. (2015). Character-level Convolutional Networks for Text Classification. *NeurIPS*.
- Jauhiainen, T., Lui, M., Zampieri, M., Baldwin, T. & Lindén, K. (2019). Automatic Language Identification in Texts: A Survey. *Journal of Artificial Intelligence Research*, 65.
- Conjunto de datos: [papluca/language-identification](https://huggingface.co/datasets/papluca/language-identification) (Hugging Face).